# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hussainhhgh/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
!git clone https://github.com/Hussainhhgh/flyrank-ml-internship.git 2>/dev/null


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

**Distributions:** Most key fields show heavy right-skew — search_volume (skew=26.0) and ctr (skew=17.4) are the most extreme, meaning a small number of high-value outliers pull the mean well above the median (e.g. ctr mean=0.51 vs median=0.07). avg_position is also notably skewed (1.98), consistent with most pages ranking outside the top spots while a smaller cluster sits near position 0-3. These heavy tails mean mean-based comparisons can be misleading; bucket/tier-based comparisons (used throughout this audit) are more robust.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
df = pd.read_csv('flyrank-ml-internship/data/raw/content_refresh_anonymized.csv')

key_fields = ['content_age_days', 'avg_position', 'ctr', 'impressions_90d', 'engagement_rate', 'search_volume']
print(df[key_fields].describe())
print("\nSkew check (heavy tails):")
print(df[key_fields].skew())

       content_age_days  avg_position           ctr  impressions_90d  \
count       30000.00000   30000.00000  30000.000000     30000.000000   
mean          256.16780      16.34238      0.510733      5200.366300   
std           132.70793      15.21679      3.279162     16838.019547   
min            90.00000       0.00000      0.000000         1.000000   
25%           132.00000       6.20000      0.000000        81.000000   
50%           236.00000      10.80000      0.070000       731.000000   
75%           333.00000      22.30000      0.290000      3615.250000   
max           564.00000     245.00000    100.000000    517715.000000   

       engagement_rate  search_volume  
count     30000.000000   27532.000000  
mean          2.534520     158.882391  
std           8.310096    1518.270825  
min           0.000000       0.000000  
25%           0.000000       0.000000  
50%           0.000000      10.000000  
75%           1.350000      20.000000  
max         100.000000   74000.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

**Signal test verdicts:**

**Signal 1 (freshness_tier): MIXED.** Decline rate rises from 0-30 days (51.1%) to 31-90 (58.9%) to 91-180 (61.1%), but reverses at 181+ (47.1%) — though that oldest bucket is small (n=174) and may be unreliable.

**Signal 2 (position_tier): OPPOSITE.** Decline risk does not rise monotonically with worse position — top_3 (24.1%) and deep (34.4%) show the lowest decline rates, while striking (61.0%), page_1 (57.0%), and page_3_5 (56.2%) — the middle of the pack — show the highest. This is the same reversal found in Week 4 and directly shaped the baseline rule design.

**Signal 3 (content_type): CONFIRMED — a real, distinct pattern.** Feedly articles decline far less often (28.7%, n=2,096) than comparison articles (57.2%, n=697) or keyword articles (56.1%, n=27,207). This is a genuinely new finding not surfaced in earlier weeks — content_type itself is informative and could be a useful additional feature.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Signal 1: freshness_tier vs decline rate (repeat from ML-07 for consistency)
print("=== Signal 1: freshness_tier ===")
s1 = df.groupby('freshness_tier').agg(n=('content_id','count'), pct_declining=('trend_direction', lambda x: (x=='down').mean()))
print(s1)
print("Verdict: MIXED (directional in first 3 tiers, reverses at 181+, per Week 4 finding)")

# Signal 2: position_tier vs decline rate
print("\n=== Signal 2: position_tier ===")
s2 = df.groupby('position_tier').agg(n=('content_id','count'), pct_declining=('trend_direction', lambda x: (x=='down').mean()))
print(s2)
print("Verdict: OPPOSITE (risk concentrated mid-pack, not at extremes, per Week 4 finding)")

# Signal 3: content_type vs decline rate (new signal for this deeper audit)
print("\n=== Signal 3: content_type ===")
s3 = df.groupby('content_type').agg(n=('content_id','count'), pct_declining=('trend_direction', lambda x: (x=='down').mean()))
print(s3)


=== Signal 1: freshness_tier ===
                    n  pct_declining
freshness_tier                      
0-30            20480       0.511377
181+              174       0.471264
31-90             175       0.588571
91-180           9171       0.611057
Verdict: MIXED (directional in first 3 tiers, reverses at 181+, per Week 4 finding)

=== Signal 2: position_tier ===
                   n  pct_declining
position_tier                      
deep            1319       0.344200
page_1         11814       0.569663
page_3_5        7242       0.561585
striking        7304       0.609529
top_3           2321       0.240844
Verdict: OPPOSITE (risk concentrated mid-pack, not at extremes, per Week 4 finding)

=== Signal 3: content_type ===
                        n  pct_declining
content_type                            
comparison article    697       0.572453
feedly article       2096       0.286737
keyword article     27207       0.560959


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

**Flag-linked test — CTR-fix flag assumption:** FlyRank's "needs CTR fix" flag assumes high impressions combined with low CTR signals a fixable snippet/title problem, not a content-quality problem. Testing this: pages with above-median impressions and below-median CTR show a 67.5% decline rate (n=3,309), compared to 52.6% for all other pages (n=26,691) — a 14.9-point gap. Verdict: CONFIRMED. The data supports the flag's underlying assumption — this specific combination of high visibility with poor click capture is meaningfully associated with decline, which is exactly the pattern the flag is designed to catch.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("=== Flag-linked test: CTR-fix flag assumption ===")
print("FlyRank's 'needs CTR fix' flag assumes: high impressions + low CTR = fixable snippet problem")

# Test: does low CTR at high impressions actually correlate with decline?
high_imp_low_ctr = df[(df['impressions_90d'] > df['impressions_90d'].median()) & (df['ctr'] < df['ctr'].median())]
rest = df[~df.index.isin(high_imp_low_ctr.index)]

print(f"\nHigh-impression + low-CTR pages: n={len(high_imp_low_ctr)}, pct_declining={high_imp_low_ctr['trend_direction'].eq('down').mean():.3f}")
print(f"All other pages: n={len(rest)}, pct_declining={rest['trend_direction'].eq('down').mean():.3f}")

=== Flag-linked test: CTR-fix flag assumption ===
FlyRank's 'needs CTR fix' flag assumes: high impressions + low CTR = fixable snippet problem

High-impression + low-CTR pages: n=3309, pct_declining=0.675
All other pages: n=26691, pct_declining=0.526


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

**What this means in practice:** A content team using flag-based triage should trust the CTR-fix logic — it holds up under testing (67.5% vs 52.6% decline rate). But triage should not default to "worst position = highest priority," since position risk is concentrated mid-pack, not at the extremes. Content type is an underused signal: Feedly-sourced articles are meaningfully more stable than other types and could inform both prioritization and future model features.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Practical takeaway for a content team:")
print("""
1. The CTR-fix flag's core assumption holds: pages with strong visibility but weak click capture
   decline nearly 15 points more often than other pages — worth prioritizing for snippet/title review.
2. Position alone is a poor standalone signal (OPPOSITE result) — mid-pack pages are actually the
   highest-risk group, not deep or unranked pages, so triage shouldn't default to 'worst position first'.
3. Content type matters more than expected — Feedly-sourced articles are structurally more stable
   than keyword or comparison articles, suggesting content_type is worth adding as a model feature
   in future iterations, not just a background variable.
""")

Practical takeaway for a content team:

1. The CTR-fix flag's core assumption holds: pages with strong visibility but weak click capture 
   decline nearly 15 points more often than other pages — worth prioritizing for snippet/title review.
2. Position alone is a poor standalone signal (OPPOSITE result) — mid-pack pages are actually the 
   highest-risk group, not deep or unranked pages, so triage shouldn't default to 'worst position first'.
3. Content type matters more than expected — Feedly-sourced articles are structurally more stable 
   than keyword or comparison articles, suggesting content_type is worth adding as a model feature 
   in future iterations, not just a background variable.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.